In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

In [11]:
vgg16 = models.vgg16(pretrained=True)

# Print the architecture of VGG-16
print(vgg16)

/home/shrestha/.conda/envs/stargan-v2/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/shrestha/.conda/envs/stargan-v2/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
class VGGDecoder(torch.nn.Module):
    def __init__(self):
        super(VGGDecoder, self).__init__()
        self.decoder_layers = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(512, 512, kernel_size=3, stride=2, padding=1, output_padding=1), # Upsample
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(512, 512, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(512, 512, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1), # Upsample
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(256, 256, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(256, 256, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1), # Upsample
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(128, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1), # Upsample
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(64, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.ConvTranspose2d(64, 3, kernel_size=3, stride=1, padding=1),
            torch.nn.Sigmoid()  # Assuming output should be in the range [0, 1]
        )

    def forward(self, x):
        return self.decoder_layers(x)


class Generator(nn.Module):
    def __init__(self, img_size=128, latent_dim=64):
        super(Generator, self).__init__()
        self.style_vector_creator = CreateStyleVector()
        self.decoder = VGGDecoder()
        self.fc1 = nn.Linear(32*32*32, 512)

        self.norm1 = AdaIN(256,64)
        self.norm2 = AdaIN(256,256)
        self.norm3 = AdaIN(256,512)
        self.norm4 = AdaIN(256,512)
        self.norm5 = AdaIN(256,512)

        self.encode1 = nn.Sequential(*list(vgg16.features.children())[:6])

        self.encode2 = nn.Sequential(*list(vgg16.features.children())[6:12])

        self.encode3 = nn.Sequential(*list(vgg16.features.children())[12:18])

        self.encode4 = nn.Sequential(*list(vgg16.features.children())[18:24])

        self.encode5 = nn.Sequential(*list(vgg16.features.children())[24:])
        
        self.fc2 = nn.Sequential(*list(vgg16.classifier.children()))

        self.final_layer = nn.Sequential(
            nn.ConvTranspose2d(16, 3, 3, 1, 1),  # Adjust input and output channels to match original input
            nn.Sigmoid()
        )

    
    def forward(self, img1, img2):
        style_img = self.style_vector_creator(img2)
        
        lat_vector = self.fc1(style_img.reshape(style_img.size(0),-1))
   
        # Encoding
        x = self.encode1(img1)
        
        x = self.norm1(x,lat_vector)
        
        x = self.encode2(x)
        x = self.norm2(x,lat_vector)
        x = self.encode3(x)
        x = self.norm3(x,lat_vector)
        x = self.encode4(x)
        x = self.norm4(x,lat_vector)
        x = self.encode5(x)
        x = self.norm5(x,lat_vector)
        x = x.reshape(x.size(0), -1)
        x = self.fc2(x)
        
        decoded_output = self.decoder(x)
        
        return x